# PDF Q&A Chatbot — RAG Pipeline
> **Stack:** LangChain · ChromaDB · SentenceTransformers · HuggingFace · Streamlit

## Step 1 — Install Dependencies

In [ ]:
# Uncomment and run once if packages are not installed
# !pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypdf python-dotenv huggingface_hub streamlit

## Step 2 — Load Environment Variables

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("CHROMA_HUGGINGFACE_API_KEY")

if not HF_TOKEN:
    raise ValueError("HuggingFace token not found. Add it to your .env file.")

print("Token loaded successfully.")

## Step 3 — Load PDF and Split into Chunks

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("genai_text.pdf")
pages  = loader.load()
print(f"Pages loaded: {len(pages)}")

full_text = " ".join([page.page_content for page in pages])

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_text(full_text)

print(f"Chunks created: {len(chunks)}")
print(f"\nSample:\n{chunks[0]}")

## Step 4 — Build ChromaDB Vector Store (Persistent)

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="pdf_qa",
    embedding_function=embedding_fn
)

if collection.count() == 0:
    ids       = [str(i) for i in range(len(chunks))]
    metadatas = [{"source": "genai_text.pdf", "chunk_index": i} for i in range(len(chunks))]
    collection.add(ids=ids, documents=chunks, metadatas=metadatas)
    print(f"Indexed {collection.count()} chunks to disk.")
else:
    print(f"Collection already exists with {collection.count()} chunks.")

## Step 5 — Retriever Function

In [ ]:
def retrieve_context(query: str, top_k: int = 5) -> str:
    results        = collection.query(query_texts=[query], n_results=top_k)
    matched_chunks = results["documents"][0]
    return "\n\n---\n\n".join(matched_chunks)


print("Retriever ready.")
print(retrieve_context("What are types of Generative AI models?"))

## Step 6 — LLM Setup

In [ ]:
from huggingface_hub import InferenceClient

llm_client = InferenceClient(api_key=HF_TOKEN)


def ask_llm(prompt: str, max_tokens: int = 400) -> str:
    response = llm_client.chat.completions.create(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


print(ask_llm("Reply with only: LLM is working."))

## Step 7 — RAG Pipeline

In [ ]:
def rag_answer(question: str) -> str:
    context = retrieve_context(question, top_k=5)

    prompt = f"""You are a helpful assistant answering questions based on a document about Generative AI.
Use the context below to answer as completely as possible.
If related information exists in the context, use it to form a helpful answer.
Only say "I don't know based on the provided document" if the context is completely irrelevant.

Context:
{context}

Question: {question}

Answer:"""

    return ask_llm(prompt, max_tokens=400)


question = "What is the history of Generative AI?"
print(f"Q: {question}\n")
print(f"A: {rag_answer(question)}")

## Step 8 — Launch Streamlit UI
> Run the cell below to start the Streamlit app in a separate terminal.

In [ ]:
import subprocess
subprocess.Popen(["streamlit", "run", "app.py"])
print("Streamlit app launched. Open http://localhost:8501 in your browser.")